<a href="https://colab.research.google.com/github/MouseLand/cellpose/blob/main/notebooks/run_Cellpose-SAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Run Cellpose-SAM

Adapted from Marius Pachitariu, Michael Rariden, Carsen Stringer and the notebook by Pradeep Rajasekhar, inspired by the [ZeroCostDL4Mic notebook series](https://github.com/HenriquesLab/ZeroCostDL4Mic/wiki)

[paper](https://www.biorxiv.org/content/10.1101/2025.04.28.651001v1) | [code](https://github.com/MouseLand/cellpose)

### Make sure you are in the correct environment


In [9]:
# Check GPU and instantiate model - will download weights.
import numpy as np
from cellpose import models, core, io, plot, utils
from pathlib import Path
from tqdm import trange
import matplotlib.pyplot as plt
import cv2 as cv 
import tifffile as tf
%matplotlib inline
from natsort import natsorted

io.logger_setup() # run this to get printing of progress

#Check if GPU access


2025-08-30 19:50:34,195 [INFO] WRITING LOG OUTPUT TO /home/mattiazzilab/.cellpose/run.log
2025-08-30 19:50:34,195 [INFO] 
cellpose version: 	4.0.6 
platform:       	linux 
python version: 	3.10.0 
torch version:  	2.7.0+cu126


(<Logger cellpose.io (INFO)>,
 PosixPath('/home/mattiazzilab/.cellpose/run.log'))

In [10]:
if core.use_gpu()==False:
  raise ImportError("No GPU access, change your runtime")

model = models.CellposeModel(gpu=True)

2025-08-30 19:50:34,216 [INFO] ** TORCH CUDA version installed and working. **
2025-08-30 19:50:34,217 [INFO] ** TORCH CUDA version installed and working. **
2025-08-30 19:50:34,217 [INFO] >>>> using GPU (CUDA)
2025-08-30 19:50:35,114 [INFO] >>>> loading model /home/mattiazzilab/.cellpose/models/cpsam


Input directory with your images:
- Note - For best accuracy and runtime performance, resize images so cells are less than 100 pixels across

In [11]:
#Inputs
dirfolder = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/MRC-5_MAX_SUM_PROJ/"
dirfolder = Path(dirfolder)
if not dirfolder.exists():
  raise FileNotFoundError("directory does not exist")


dirlist = [dir for dir in dirfolder.glob("*")]
display(dirlist)

from cellpose_functions import *

# *** change to your image extension ***
image_ext = ".tif"
#nchannels = 4



[PosixPath('/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/MRC-5_MAX_SUM_PROJ/20250410_rep06'),
 PosixPath('/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/MRC-5_MAX_SUM_PROJ/20250501_rep07'),
 PosixPath('/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/MRC-5_MAX_SUM_PROJ/20240326_rep02'),
 PosixPath('/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/MRC-5_MAX_SUM_PROJ/for_figs'),
 PosixPath('/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/MRC-5_MAX_SUM_PROJ/20250328_rep05'),
 PosixPath('/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/MRC-5_MAX_SUM_PROJ/20241112_rep04'),
 PosixPath('/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/MRC-5_MAX_SUM_PROJ/20241018_rep03'),
 PosixPath('/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/MRC-5_MAX_SUM_PROJ/20240313_rep01')]

In [12]:
img_files_test = load_sorted_directory_list(dirlist[0])
maskdir = dirlist[0] / "a_testmasks"
maskdir.mkdir(exist_ok=True)

display(img_files_test)
print(maskdir)

[PosixPath('/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/MRC-5_MAX_SUM_PROJ/20250410_rep06/MAX_ch1-r01c02f01.tif'),
 PosixPath('/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/MRC-5_MAX_SUM_PROJ/20250410_rep06/MAX_ch2-r01c02f01.tif'),
 PosixPath('/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/MRC-5_MAX_SUM_PROJ/20250410_rep06/MAX_ch3-r01c02f01.tif'),
 PosixPath('/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/MRC-5_MAX_SUM_PROJ/20250410_rep06/MAX_ch4-r01c02f01.tif'),
 PosixPath('/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/MRC-5_MAX_SUM_PROJ/20250410_rep06/MAX_ch1-r01c02f02.tif'),
 PosixPath('/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/MRC-5_MAX_SUM_PROJ/20250410_rep06/MAX_ch2-r01c02f02.tif'),
 PosixPath('/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/MRC-5_MAX_SUM_PROJ/20250410_rep06/MAX_ch3-r01c02f02.tif'),
 PosixPath('/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative

/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/MRC-5_MAX_SUM_PROJ/20250410_rep06/a_testmasks


In [13]:
for dir in dirlist:
    maskdir = dir / "newmasks"
    maskdir.mkdir(exist_ok=True)
    #nchannels = 4
    if dir.name == "for_figs":#the folders that are already done
        continue
    else:
        if dir.name == "20241112_rep04":
            nchannels = 3
        else:
            nchannels = 4
        print("writing to: " + str(maskdir))
        files = load_sorted_directory_list(dir)
        save_mask_folder(files,maskdir,v2=True)



writing to: /mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/MRC-5_MAX_SUM_PROJ/20250410_rep06/newmasks
2025-08-30 19:50:44,994 [INFO] WRITING LOG OUTPUT TO /home/mattiazzilab/.cellpose/run.log
2025-08-30 19:50:44,994 [INFO] 
cellpose version: 	4.0.6 
platform:       	linux 
python version: 	3.10.0 
torch version:  	2.7.0+cu126
2025-08-30 19:50:44,995 [INFO] ** TORCH CUDA version installed and working. **
2025-08-30 19:50:44,995 [INFO] ** TORCH CUDA version installed and working. **
2025-08-30 19:50:44,995 [INFO] >>>> using GPU (CUDA)
2025-08-30 19:50:45,886 [INFO] >>>> loading model /home/mattiazzilab/.cellpose/models/cpsam


  0%|          | 0/3120 [00:00<?, ?it/s]

2025-08-30 19:50:48,282 [WARNING] Resizing is depricated in v4.0.1+


  0%|          | 1/3120 [00:02<1:53:12,  2.18s/it]

0.5255706591433701
2025-08-30 19:50:51,028 [WARNING] Resizing is depricated in v4.0.1+


...

0.8078534031413612
2025-08-30 19:50:55,993 [WARNING] Resizing is depricated in v4.0.1+


  0%|          | 4/3120 [00:09<2:08:15,  2.47s/it]

0.47844620877071453
2025-08-30 19:50:58,209 [INFO] No cell pixels found.
2025-08-30 19:50:58,366 [WARNING] no seeds found in get_masks_torch - no masks found.
2025-08-30 19:50:58,366 [WARNING] Resizing is depricated in v4.0.1+


  0%|          | 4/3120 [00:12<2:39:08,  3.06s/it]


IndexError: index 0 is out of bounds for axis 0 with size 0

[INFO] WRITING LOG OUTPUT TO /home/mattiazzilab/.cellpose/run.log
Note: remember https://pmc.ncbi.nlm.nih.gov/articles/PMC8560386/#sec5 for mitochondria

### set up the correct order for the image filenames - sort by location first, then channel
these functions moved to cellpose_functions
```python
def file_sort_key(filename):
  parts = filename.split("-")
  channel = parts[0][-1:] # get the last character of the first part
  location = parts[1]
  return (location,channel)

def plate_location(filename):
  parts = filename.split("-")
  pre_location = parts[1]
  location = pre_location.split(".")[0] # get the first part of the second part
  return location
  
#list all files
def sort_files(dir, image_ext):
  if not dir.exists():
    raise FileNotFoundError("directory does not exist")
  files = sorted([f for f in dir.glob("*"+image_ext) if "_masks" not in f.name and "_flows" not in f.name and "SUM" not in f.name],
                           key=lambda x: file_sort_key(x.name))```
 # sort by number in filename
  if(len(files)==0):
    raise FileNotFoundError("no image files found, did you specify the correct folder and extension?")
  else:
    return files
  
def print_files(files):
  for f in files:
    print(f.name)

def group_files_by_channel(files, nchannels=4):
  grouped = []
  for i in range(0,len(files),nchannels):
    grouped.append(files[i:i+nchannels])
  return grouped

def print_grouped_files(grouped):
  for i in range(len(grouped)):
    print(f"\n Group {i+1} of {len(grouped)}")
    for j in range(len(grouped[i])):
      item = grouped[i][j]
      print(" "+ item.name)
```

In [ ]:
files = img_files_test 
#files = sort_files(dir, image_ext)
grouped_files = group_files_by_channel(files)

#print_files(files)
print_grouped_files(grouped_files)
print(plate_location(files[-1].name))

print(get_nchannels(files))

## Run Cellpose-SAM on one image in folder

Here are some of the parameters you can change:

* ***flow_threshold*** is  the  maximum  allowed  error  of  the  flows  for  each  mask.   The  default  is 0.4.
    *  **Increase** this threshold if cellpose is not returning as many masks as you’d expect (or turn off completely with 0.0)
    *   **Decrease** this threshold if cellpose is returning too many ill-shaped masks.

* ***cellprob_threshold*** determines proability that a detected object is a cell.   The  default  is 0.0.
    *   **Decrease** this threshold if cellpose is not returning as many masks as you’d expect or if masks are too small
    *   **Increase** this threshold if cellpose is returning too many masks esp from dull/dim areas.

* ***tile_norm_blocksize*** determines the size of blocks used for normalizing the image. The default is 0, which means the entire image is normalized together.
  You may want to change this to 100-200 pixels if you have very inhomogeneous brightness across your image.



In [ ]:
image_set_index = 774
in_channels = load_image_set(grouped_files[image_set_index])
set_name = get_image_set_name(grouped_files[image_set_index])
print("Set name: ", set_name)
display(in_channels)

img1 = img_preprocessing(in_channels)
img2 = img_rescaled(img1, factor=0.25)
img_unprocessed = get_multichannel_img_normalized(in_channels)
img_unprocessed_2 = img_rescaled(img_unprocessed, factor=0.25)

img1_v2 = img_preprocessing_v2(in_channels)
img2_v2 = img_rescaled(img1_v2, factor=0.25)

tf.imshow(img2)
# tf.imshow(img1)
tf.imshow(img2_v2)

In [ ]:
img = img2
from skimage import feature, filters   
cell_masks = segment_cell(img,model)
nuc_masks = segment_nuclei(img,model)
cell_v2_masks = segment_cell_v2(img2_v2,model)
nuc_v2_masks = segment_nuclei_v2(img2_v2, model)

print(maskdir)

save_masks(set_name, cell_masks, outdir=maskdir, image_ext=image_ext, mask_type="cell")
save_masks(
    set_name, nuc_masks, outdir=maskdir, image_ext=image_ext, mask_type="nuclei"
)
save_masks(
    set_name, cell_v2_masks, outdir=maskdir, image_ext=image_ext, mask_type="cell_v2"
)
save_masks(set_name, nuc_v2_masks, outdir=maskdir, image_ext=image_ext, mask_type="nuclei_v2")


### Channel Selection
If you have a fluroescent image with multiple stains, you should choose one channel with a cytoplasm/membrane stain, one channel with a nuclear stain, and set the third channel to None. Choosing multiple channels may produce segmentaiton of all the structures in the image. If you have retrained the model on your data with a thrid stain (described below), you can run segmentation with all channels.

In [ ]:
# img = io.imread(files[0])
img = img2
def segment_cell_tweaking(
    img,
    model,
    show_plot=True,
    flow_threshold=0.6,
    cellprob_threshold=-1,
    tile_norm_blocksize=100,
    diameter=60,
    min_size=500,
    max_size_frac=0.85,  # keep masks up to 70% of image size
    niter=1000,
):
    from skimage import filters, morphology, exposure

    gfp = img[:, :, 0]  # combine the ch1 and ch2 images to help cellpose out a bit
    rfp = img[:, :, 1]
    dapi = img[:, :, 2]  # save ch3 for later

    img_combo = gfp + rfp
    img_combo = img_01_normalization(img_combo)
    #tf.imshow(img_combo, cmap="viridis")
    # smooth image and improve outline (sigma is the gaussian kernel)
    img_combo = filters.gaussian(img_combo, sigma=1)
    #Can also use unsharp mask, but it tends to chop the outlines too short img_combo = filters.unsharp_mask(img_combo, radius=0.5, amount=2)

    # stack the images
    img_selected_channels = np.stack([img_combo, dapi], axis=-1)

    masks, flows, styles = model.eval(
        img_selected_channels,
        batch_size=64,
        niter=niter,
        diameter=diameter,
        flow_threshold=flow_threshold,
        cellprob_threshold=cellprob_threshold,
        normalize={"tile_norm_blocksize": tile_norm_blocksize},
        max_size_fraction=max_size_frac,
        min_size=min_size
    )
    masks = utils.fill_holes_and_remove_small_masks(masks, min_size=min_size)
    masks = utils.dilate_masks(masks, n_iter=2)
    print(utils.size_distribution(masks))
    # plot if true
    if show_plot:
        fig = plt.figure(figsize=(12, 5))
        plot.show_segmentation(fig, img_selected_channels, masks, flows[0])
        plt.tight_layout()
        plt.show()
    return masks


def segment_nuclei_tweaking(
    orig_img,
    model,
    show_plot=True,
    flow_threshold=0.5,
    cellprob_threshold=1,
    tile_norm_blocksize=100,
    min_size=400,
    max_size_frac=0.4,
    diameter=None,
    niter=None,
):
    from skimage import morphology, filters

    img = orig_img[:, :, 2]  # get the DAPI channel

    # remove speckle-shaped autofluor
    bg2 = morphology.white_tophat(img, morphology.disk(3))
    img = img - bg2
    img = morphology.closing(img, morphology.disk(2.5))
    
    # sharpen image and improve outline (radius is the gaussian kernel)
    img = img_01_normalization(img)
    img = filters.gaussian(img, sigma=2)
    #tf.imshow(img, cmap="plasma")
    #img = filters.unsharp_mask(img, radius=1, amount=2)
    masks, flows, styles = model.eval(
        img,
        batch_size=64,
        diameter=diameter,
        niter=niter,
        flow_threshold=flow_threshold,
        cellprob_threshold=cellprob_threshold,
        normalize={"tile_norm_blocksize": tile_norm_blocksize},
        max_size_fraction=max_size_frac,
    )
    # dilate before removing the ones touching edges to catch the stragglers
    masks = utils.dilate_masks(masks, n_iter=1)
    masks_removed_edges = utils.remove_edge_masks(masks)
    masks_removed_edges = utils.fill_holes_and_remove_small_masks(
        masks_removed_edges, min_size=min_size
    )

    if show_plot:
        fig = plt.figure(figsize=(12, 5))
        plot.show_segmentation(fig, img, masks_removed_edges, flows[0])
        plt.tight_layout()
        plt.show()
    return masks_removed_edges

cell_masks = segment_cell_tweaking(img,model)
cell_masks_2 = segment_cell_tweaking(img2_v2, model)
cell_masks_OG = segment_cell(img, model)
nuc_masks = segment_nuclei_tweaking(img, model)
nuc_masks_2 = segment_nuclei_tweaking(img2_v2, model)
# save_masks(set_name+"tweaking", cell_masks_2, outdir=maskdir, image_ext=image_ext)
# save_masks(set_name, nuc_masks, outdir=maskdir, image_ext=image_ext, mask_type="nuclei")


## Run Cellpose-SAM on folder of images

if you have many large images, you may want to run them as a loop over images
See `cellpose_functions` to run this
 moved to `save_mask_folder() `


### loop for all files in the group in the directory
```python
for i in trange(len(grouped_files)):
    file_group = grouped_files[i]
    img_set = load_image_set(file_group)
    img_set_name = get_image_set_name(file_group)
    print("Set name: ", set_name)
    
    stacked_img = img_preprocessing(img_set)
    rescaled_img = img_rescaled(stacked_img, factor=0.25)
    
    cell_masks = segment_cell(rescaled_img, show=False)
    nuc_masks = segment_nuclei(rescaled_img, show=False) 
    
    save_masks(img_set_name, cell_masks, image_ext=image_ext)
    save_masks(img_set_name, nuc_masks, image_ext=image_ext, mask_type="nuclei") 
```